In [ ]:
# ==========================================================
# EVALUATION COMPLETE (MULTICLASSE + BINAIRE)
#   - pour CustomPooling (streaming)
#   - pour AvgPool temporal-only (streaming)
# ==========================================================

import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    f1_score,
    classification_report,
    balanced_accuracy_score,
    roc_curve,
    auc
)

class ModelAvgPoolingTemporal(nn.Module):
    def __init__(self, n_features=82, d_hist=20, n_classes=15):
        super().__init__()

        self.conv1 = nn.Conv2d(1, 16, (1, 3))
        self.pool1 = nn.AvgPool2d(kernel_size=(1, 3), stride=(1, 3))

        self.conv2 = nn.Conv2d(16, 16, (1, 2))
        self.pool2 = nn.AvgPool2d(kernel_size=(1, 2), stride=(1, 2))

        self.conv3 = nn.Conv2d(16, 16, (1, 2))

        self.flatten = nn.Flatten()

        with torch.no_grad():
            dummy = torch.zeros(1, 1, n_features, d_hist)
            dummy = torch.relu(self.conv1(dummy))   # -> [1,16,82,18]
            dummy = self.pool1(dummy)               # -> [1,16,82,6]
            dummy = torch.relu(self.conv2(dummy))   # -> [1,16,82,5]
            dummy = self.pool2(dummy)               # -> [1,16,82,2]
            dummy = torch.relu(self.conv3(dummy))   # -> [1,16,82,1]
            flatten_size = dummy.numel()            # 1312

        self.fc = nn.Sequential(
            nn.Linear(flatten_size, 32),
            nn.ReLU(),
            nn.Linear(32, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, n_classes)
        )

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = self.pool1(x)
        x = torch.relu(self.conv2(x))
        x = self.pool2(x)
        x = torch.relu(self.conv3(x))
        x = self.flatten(x)
        return self.fc(x)



# ==========================================================
# MAPPING ORIGINAL
# ==========================================================
original_labels = {
    0: "BENIGN",
    1: "Bot",
    2: "DDoS",
    3: "DoS GoldenEye",
    4: "DoS Hulk",
    5: "DoS Slowhttptest",
    6: "DoS slowloris",
    7: "FTP-Patator",
    8: "Heartbleed",
    9: "Infiltration",
    10: "PortScan",
    11: "SSH-Patator",
    12: "Web Attack – Brute Force",
    13: "Web Attack – Sql Injection",
    14: "Web Attack – XSS"
}

# ==========================================================
# REGROUPEMENT
# ==========================================================
def regroup_classes(label_id):
    if label_id in [3, 4, 5, 6]:
        return "DoS"
    elif label_id in [12, 13, 14]:
        return "WebAttack"
    elif label_id in [7, 11]:
        return "Patator"
    else:
        return original_labels[int(label_id)]


# ==========================================================
# Dataset TEST streaming (no concat)
# ==========================================================
class NPYSplitDataset(Dataset):
    def __init__(self, folder, y_path, n_files=20, prefix="X_input_", mmap=True):
        self.folder = folder
        self.n_files = n_files
        self.paths = [os.path.join(folder, f"{prefix}{i}.npy") for i in range(n_files)]
        self.Y = np.load(y_path).astype(np.int64, copy=False)

        self._arrays = []
        self._lengths = []
        for p in self.paths:
            arr = np.load(p, mmap_mode="r" if mmap else None)
            self._arrays.append(arr)
            self._lengths.append(arr.shape[0])

        total = sum(self._lengths)
        assert total == len(self.Y), f"Total X ({total}) != len(Y) ({len(self.Y)})"

        self._offsets = np.cumsum([0] + self._lengths)

    def __len__(self):
        return len(self.Y)

    def _locate(self, gidx):
        f = np.searchsorted(self._offsets, gidx, side="right") - 1
        local = gidx - self._offsets[f]
        return int(f), int(local)

    def __getitem__(self, idx):
        f, j = self._locate(idx)
        x = self._arrays[f][j]  # [F,W]
        y = self.Y[idx]
        x_t = torch.from_numpy(np.array(x, copy=False)).float().unsqueeze(0)  # [1,F,W]
        y_t = torch.tensor(int(y), dtype=torch.long)
        return x_t, y_t


# ==========================================================
# EVALUATION
# ==========================================================
@torch.no_grad()
def evaluate_model_from_loader(model, loader):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()

    all_preds, all_probs, all_labels = [], [], []

    for xb, yb in tqdm(loader, desc="Evaluation"):
        xb = xb.to(device, non_blocking=True)  # [B,1,F,W]
        outputs = model(xb)

        probs = torch.softmax(outputs, dim=1)
        preds = torch.argmax(probs, dim=1)

        all_preds.append(preds.detach().cpu().numpy())
        all_probs.append(probs.detach().cpu().numpy())
        all_labels.append(yb.numpy())

    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)
    all_probs = np.concatenate(all_probs)

    # ======================================================
    # MULTICLASSE ORIGINALE
    # ======================================================
    print("\n========== MULTICLASSE ORIGINALE ==========")
    print("Accuracy :", accuracy_score(all_labels, all_preds))
    print("F1 weighted:", f1_score(all_labels, all_preds, average="weighted"))
    print("Balanced accuracy:", balanced_accuracy_score(all_labels, all_preds))
    print("\nRapport détaillé :")
    print(classification_report(all_labels, all_preds, digits=4))

    # ======================================================
    # MULTICLASSE REGROUPEE
    # ======================================================
    print("\n========== MULTICLASSE REGROUPEE ==========")
    y_true_group = np.array([regroup_classes(l) for l in all_labels])
    y_pred_group = np.array([regroup_classes(l) for l in all_preds])

    print("Accuracy :", accuracy_score(y_true_group, y_pred_group))
    print("F1 weighted:", f1_score(y_true_group, y_pred_group, average="weighted"))
    print("Balanced accuracy:", balanced_accuracy_score(y_true_group, y_pred_group))
    print("\nRapport détaillé regroupé :")
    print(classification_report(y_true_group, y_pred_group, digits=4))

    cm_group = confusion_matrix(y_true_group, y_pred_group)
    cm_group_norm = cm_group.astype(float) / cm_group.sum(axis=1, keepdims=True)
    cm_group_norm = np.nan_to_num(cm_group_norm)

    plt.figure(figsize=(10, 8))
    disp = ConfusionMatrixDisplay(cm_group_norm)
    disp.plot(cmap="Blues", values_format=".2f")
    plt.title("Matrice regroupée (%) - Normalisée par ligne")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    # ======================================================
    # BINAIRE
    # ======================================================
    print("\n========== BINAIRE ==========")
    y_true_bin = (all_labels != 0).astype(int)
    y_pred_bin = (all_preds != 0).astype(int)

    print("Accuracy :", accuracy_score(y_true_bin, y_pred_bin))
    print("F1 :", f1_score(y_true_bin, y_pred_bin))
    print("Balanced accuracy :", balanced_accuracy_score(y_true_bin, y_pred_bin))

    cm_bin = confusion_matrix(y_true_bin, y_pred_bin)
    cm_bin_norm = cm_bin.astype(float) / cm_bin.sum(axis=1, keepdims=True)
    cm_bin_norm = np.nan_to_num(cm_bin_norm)

    plt.figure(figsize=(5, 5))
    disp = ConfusionMatrixDisplay(cm_bin_norm, display_labels=["Normal", "Attaque"])
    disp.plot(cmap="Reds", values_format=".2f")
    plt.title("Matrice Binaire (%)")
    plt.tight_layout()
    plt.show()

    attack_probs = all_probs[:, 1:].sum(axis=1)
    fpr, tpr, _ = roc_curve(y_true_bin, attack_probs)
    roc_auc = auc(fpr, tpr)

    plt.figure()
    plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.4f}")
    plt.plot([0, 1], [0, 1], "--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve (Binaire)")
    plt.legend()
    plt.show()


# ==========================================================
# CONFIG PATHS (TEST)
# ==========================================================
TEST_FOLDER = "./X_input_split_test"
Y_TEST_PATH = "./X_input_split_test/Y_test.npy"
N_FILES = 20

# Si tes dimensions diffèrent, ajuste :
N_FEATURES = 82
D_HIST = 20
N_CLASSES = 15

# ==========================================================
# Build loader once
# ==========================================================
print("--------------------Chargement loader test (streaming)--------------------")
test_ds = NPYSplitDataset(TEST_FOLDER, Y_TEST_PATH, n_files=N_FILES, prefix="X_input_", mmap=True)
test_loader = DataLoader(test_ds, batch_size=512, shuffle=False, pin_memory=True, num_workers=0)

print("Test size:", len(test_ds))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# ==========================================================
# 2) Eval AvgPool temporal-only model
# ==========================================================
print("\n==================== EVAL: AvgPool Temporal ====================")
model_avg = ModelAvgPoolingTemporal(n_features=N_FEATURES, d_hist=D_HIST, n_classes=N_CLASSES).to(device)
model_avg.load_state_dict(torch.load("./cicids_models/model_avgpool_cicids.pth", map_location=device))

evaluate_model_from_loader(model_avg, test_loader)